# **IMPLEMENTACIÓN DE UN ALGORITMO GENÉTICO**

## **¿QUÉ ES UN ALGORITMO GENÉTICO?**

Es un método de optimización inspirado en la teoría de la evolución de Darwin, según la cual los individuos mejor adaptados a su entorno tienen más probabilidades de sobrevivir y dejar descendencia. Siguiendo esa idea, en vez de probar todas las soluciones posibles, partimos de una población de soluciones y dejamos que las mejores sobrevivan y se reproduzcan, generación tras generación.

Cada solución, o individuo, se representa como un cromosoma; en nuestro caso, un vector de ceros y unos. Para saber qué tan bueno es cada individuo usamos una función de aptitud (fitness), elegimos a los mejores como padres, los cruzamos para crear hijos que mezclan sus genes y, por último, aplicamos una mutación que cambia algunos genes al azar para no perder diversidad.

### **Ventajas y desventajas**

Su principal ventaja es que no necesita que el problema sea continuo ni derivable y, como cada individuo se evalúa por separado, es fácil de paralelizar. Por otro lado, no garantiza encontrar la mejor solución, puede tardar bastante y sus resultados dependen mucho de parámetros como el tamaño de la población o la probabilidad de mutación.

## **PLANTEAMIENTO DEL PROBLEMA**

Queremos predecir si una persona tiene diabetes o prediabetes con un árbol de decisión, pero no todas las 21 variables de la encuesta aportan información útil. El problema de optimización consiste en encontrar el subconjunto de variables con el que el modelo obtiene el mejor desempeño.

- **Variables de decisión:** un vector de 21 genes, uno por variable. Un 1 indica que la variable se usa para entrenar el modelo y un 0 que se descarta.
- **Función objetivo:** maximizar el F1 promedio del árbol de decisión en una validación cruzada estratificada de 3 folds. Usamos F1 porque las clases están desbalanceadas: solo el 15.29% de los registros corresponde a personas con diabetes o prediabetes.
- **Espacio de búsqueda:** existen $2^{21} \approx 2.1 \times 10^{6}$ combinaciones posibles. Una corrida de 100 generaciones evalúa unas 10100, cerca del 0.5% del total, así que el algoritmo genético debe buscar de forma dirigida en lugar de probarlas todas.

In [4]:
%pip install ucimlrepo

In [5]:
import os
import numpy as np
import random
import time
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from ucimlrepo import fetch_ucirepo

## **CARGA DE DATOS**

Usamos el dataset **CDC Diabetes Health Indicators**, el mismo del notebook de análisis exploratorio. Unimos las variables con el objetivo y eliminamos los registros duplicados, que son respuestas idénticas de la encuesta: si se conservan, un mismo registro puede quedar a la vez en entrenamiento y en prueba. De las 253680 filas iniciales quedan 229474.

In [6]:
# fetch dataset 
cdc_diabetes_health_indicators = fetch_ucirepo(id=891) 

In [7]:
X = cdc_diabetes_health_indicators.data.features 
y = cdc_diabetes_health_indicators.data.targets 

In [8]:
df = pd.concat([X, y], axis=1)

In [9]:
# df = df.drop_duplicates()

df.shape

(253680, 22)

In [10]:
df.shape

(253680, 22)

In [11]:
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

In [12]:
X.shape

(253680, 21)

## **ETAPA #1: Inicializamos la población**

En esta primer etapa, creamos un DataFrame con 100 individuos donde cada uno tiene un gen (0 o 1) por cada variable de `X`, 21 en total, que indica si esa variable se usa para entrenar el modelo.

In [13]:
def initialize_poblation(size=100):
    random_seed = 42
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    rng = np.random.default_rng(random_seed)
    poblacion = rng.integers(0, 2, size=(size, 21))
    poblacion = pd.DataFrame(poblacion)
    
    poblacion["F1"] = 0
    return poblacion

## **ETAPA #2: Función de Aptitud (FITNESS)**

Cada gen de un individuo corresponde a una variable de `X`: si vale 1, la variable se usa para entrenar un árbol de decisión y, si vale 0, se descarta. La aptitud es el F1 promedio de una validación cruzada estratificada de 3 folds.

In [14]:
def fitness(individuos: pd.DataFrame, X: pd.DataFrame, Y: pd.Series):
    individuos = individuos.drop('F1', axis=1)

    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    modelo = DecisionTreeClassifier(
        max_depth=5,
        class_weight='balanced',
        random_state=42
    )
    resultados_f1 = []

    for _, individuo in individuos.iterrows():

        # Genes en 1 = variables seleccionadas
        genes = individuo.values.astype(bool)
        columnas = X.columns[genes].tolist()

        # Sin variables no se puede entrenar el modelo
        if not columnas:
            resultados_f1.append(0.0)
            continue

        f1 = cross_val_score(modelo, X[columnas], Y, cv=folds, scoring='f1').mean()

        resultados_f1.append(f1)

    individuos['F1'] = resultados_f1

    return individuos

## **ETAPA #3: SELECCIÓN**

Ya tenemos una población de 100 modelos entrenados con los genes de los individuos creados en la etapa de Inicialización, ahora necesitamos extraer los 50 mejores individuos ya que de estos se derivarán las siguientes generaciones. Usaremos el metodo de elitismo, ordenaremos de mayor a menor la columna `F1` y tomaremos los 50 mayores.

In [15]:
def best_50(poblacion: pd.DataFrame):
    ordered_poblation = poblacion.sort_values(by="F1", ascending=False)
    top50 = ordered_poblation.head(50)
    return top50

## **ETAPA #4: CRUCE**

A partir de los 50 mejores individuos creamos 50 hijos con un cruce uniforme. Los individuos se agrupan en parejas y, para cada gen, con una probabilidad `gamma` de 0.5 los dos hijos intercambian ese gen; si no, cada uno conserva el de su padre. Así cada hijo recibe una mezcla aleatoria de los genes de ambos padres. La función devuelve una nueva población de 100 individuos: los 50 padres y sus 50 hijos.

In [16]:
def crossing(individuos: pd.DataFrame, gamma: float = 0.5):
    padres = individuos.drop('F1', axis=1)
    hijos = padres.to_numpy().copy()

    for i in range(0, len(hijos) - 1, 2):
        for j in range(hijos.shape[1]):
            # Con probabilidad gamma, los dos hijos intercambian este gen
            if random.random() <= gamma:
                hijos[i, j], hijos[i + 1, j] = hijos[i + 1, j], hijos[i, j]

    hijos = pd.DataFrame(hijos, columns=padres.columns)

    # La nueva población está formada por los 50 padres y sus 50 hijos
    nueva_poblacion = pd.concat([padres, hijos], ignore_index=True)
    nueva_poblacion['F1'] = 0

    return nueva_poblacion

## **ETAPA #5: MUTACIÓN**

Cada gen de la nueva población cambia de valor (de 0 a 1 o de 1 a 0) con una probabilidad de 0.05, es decir, cerca de un gen por individuo. Usamos una probabilidad baja porque, con genes binarios, una mutación alta convertiría la búsqueda en una selección aleatoria de variables.

In [17]:
def mutation(poblacion: pd.DataFrame, prob_mutacion: float = 0.05):
    genes = poblacion.drop('F1', axis=1).to_numpy().copy()

    for i in range(genes.shape[0]):
        for j in range(genes.shape[1]):
            # El gen cambia de 0 a 1 o de 1 a 0
            if random.random() <= prob_mutacion:
                genes[i, j] = 1 - genes[i, j]

    poblacion = pd.DataFrame(genes)
    poblacion['F1'] = 0
    return poblacion

## **ETAPA #6: IMPLEMENTACIÓN**

La función `genetic_algorithm` une todas las etapas. En cada generación seleccionamos los 50 mejores individuos, los cruzamos para obtener una nueva población de 100, aplicamos la mutación y evaluamos el fitness de todos. La función devuelve el mejor individuo y el historial del F1 por generación.

In [18]:
def genetic_algorithm(generations: int, X: pd.DataFrame, Y: pd.Series, population_size: int = 100):
    poblacion = initialize_poblation(size=population_size)
    poblacion = fitness(
        individuos=poblacion, 
        X=X, 
        Y=Y
    )
    
    historial = []

    for i in range(generations):
        top50 = best_50(poblacion=poblacion)

        new_generation = crossing(
            individuos=top50
        )
        
        new_generation = mutation(
            poblacion=new_generation
        )
        
        poblacion = fitness(
            individuos=new_generation, 
            X=X, 
            Y=Y
        )

        historial.append({
            'generacion': i + 1,
            'mejor_F1': poblacion['F1'].max(),
            'F1_promedio': poblacion['F1'].mean()
        })
        print(f"[INFO] Generación {i + 1}: mejor F1 = {poblacion['F1'].max():.4f}")

    mejor_individuo = poblacion.sort_values(by='F1', ascending=False).iloc[0]

    return mejor_individuo, pd.DataFrame(historial)

## **EJECUCIÓN**

Como referencia, calculamos el F1 del árbol de decisión entrenado con todas las variables. El algoritmo genético debería alcanzar un F1 similar o mayor usando menos variables.

In [19]:
todas_las_variables = pd.DataFrame([np.ones(X.shape[1], dtype=int)])
todas_las_variables['F1'] = 0

fitness(individuos=todas_las_variables, X=X, Y=y)['F1']

,F1
0,0.415264


Ejecutamos el algoritmo genético secuencial durante 10 generaciones y medimos su tiempo de ejecución, que usaremos para compararlo con la versión paralelizada con Dask.

In [21]:
inicio = time.perf_counter()

mejor_individuo, historial = genetic_algorithm(generations=10, X=X, Y=y)

tiempo_secuencial = time.perf_counter() - inicio
print(f'Tiempo de ejecución: {tiempo_secuencial:.1f} segundos')

[INFO] Generación 1: mejor F1 = 0.4241
[INFO] Generación 2: mejor F1 = 0.4258
[INFO] Generación 3: mejor F1 = 0.4260
[INFO] Generación 4: mejor F1 = 0.4260
[INFO] Generación 5: mejor F1 = 0.4260
[INFO] Generación 6: mejor F1 = 0.4260
[INFO] Generación 7: mejor F1 = 0.4260
[INFO] Generación 8: mejor F1 = 0.4260
[INFO] Generación 9: mejor F1 = 0.4260
[INFO] Generación 10: mejor F1 = 0.4260
Tiempo de ejecución: 994.2 segundos


In [28]:
# Guardamos el tiempo para compararlo con la versión de Dask en el notebook 03
os.makedirs('./outputs', exist_ok=True)

pd.DataFrame([{
    'version': 'Secuencial',
    'generaciones': len(historial),
    'tiempo_s': tiempo_secuencial
}]).to_csv('./outputs/tiempo_secuencial.csv', index=False)

## **RESULTADOS**

Variables seleccionadas por el mejor individuo y evolución del F1 en cada generación.

In [23]:
genes = mejor_individuo.drop('F1').values.astype(bool)
variables_seleccionadas = X.columns[genes].tolist()

print(f"F1 del mejor individuo: {mejor_individuo['F1']:.4f}")
print(f'Variables seleccionadas: {len(variables_seleccionadas)} de {X.shape[1]}')

variables_seleccionadas

F1 del mejor individuo: 0.4260
Variables seleccionadas: 12 de 21


['HighBP',
 'HighChol',
 'CholCheck',
 'BMI',
 'HeartDiseaseorAttack',
 'Fruits',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'PhysHlth',
 'Sex',
 'Education']

In [24]:
historial

,generacion,mejor_F1,F1_promedio
0,1,0.424138,0.402573
1,2,0.425768,0.411032
2,3,0.426012,0.416470
3,4,0.426012,0.419305
4,5,0.426012,0.421190
5,6,0.426012,0.419485
6,7,0.426012,0.419840
7,8,0.426012,0.419819
8,9,0.426012,0.421458
9,10,0.426012,0.421050
